# 01 — Téléchargement & Exploration des données OpenAgenda

**Objectif** : Récupérer les événements publics d'**Île-de-France** via l'API Opendatasoft, filtrer sur une fenêtre temporelle (1 an passé + événements futurs), sauvegarder les données brutes et explorer la structure des données en vue du pipeline RAG.

**Source** : `public.opendatasoft.com` — dataset `evenements-publics-openagenda`  
**API** : Opendatasoft v2.1 — pas de clé requise  
**Volume** : ~188 000 événements IDF disponibles (toutes dates confondues)

---

## 0. Imports & configuration

In [1]:
from datetime import datetime, timedelta
import json
import time

from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter
import pandas as pd
import requests

from notebooks.config import (
    DATA_RAW,
    OPENAGENDA_BASE_URL,
    OPENAGENDA_REGION,
    REFERENCE_DATE,
)

converter = DocumentConverter()


def html_to_text(html: str | None) -> str:
    if not isinstance(html, str) or html.strip() == "":
        return ""
    return converter.convert_string(html, format=InputFormat.HTML).document.export_to_markdown().strip()


DATA_RAW.mkdir(parents=True, exist_ok=True)


ref = datetime.strptime(REFERENCE_DATE, "%Y-%m-%d")
DATE_FROM = (ref - timedelta(days=365)).strftime("%Y-%m-%dT00:00:00Z")
DATE_TO = (ref + timedelta(days=365)).strftime("%Y-%m-%dT00:00:00Z")

WHERE_CLAUSE = (
    f'location_region = "{OPENAGENDA_REGION}"'
    f' AND firstdate_begin >= "{DATE_FROM}"'
    f' AND lastdate_begin <= "{DATE_TO}"'
)

print(f"Région          : {OPENAGENDA_REGION}")
print(f"Fenêtre         : {DATE_FROM} → {DATE_TO}")
print(f"Date référence  : {REFERENCE_DATE}")
print(f"Dossier sortie  : {DATA_RAW.resolve()}")
print(f"\nFiltre WHERE :\n{WHERE_CLAUSE}")

Région          : Île-de-France
Fenêtre         : 2025-06-07T00:00:00Z → 2027-06-07T00:00:00Z
Date référence  : 2026-06-07
Dossier sortie  : /Users/kilian/Documents/Repositories/OpenClassrooms/OC-Concevez-et-deployez-un-systeme-RAG/data/raw

Filtre WHERE :
location_region = "Île-de-France" AND firstdate_begin >= "2025-06-07T00:00:00Z" AND lastdate_begin <= "2027-06-07T00:00:00Z"


## 1. Appel API — récupération paginée

L'API retourne **100 enregistrements max par page** → on pagine via `offset` dans `OpenAgendaClient`.

**Filtres ODSQL appliqués** (construits par `build_where_clause`) :
- `location_region = "Île-de-France"` — filtre géographique
- `firstdate_begin >= DATE_FROM` — pas d'événements trop anciens
- `lastdate_begin <= DATE_TO` — pas d'événements trop lointains

In [2]:
PAGE_SIZE = 100
MAX_RECORDS = 10_000
POLITENESS_DELAY = 0.2

raw_events: list[dict] = []
offset = 0

while True:
    params = {"where": WHERE_CLAUSE, "limit": PAGE_SIZE, "offset": offset, "lang": "fr"}
    resp = requests.get(OPENAGENDA_BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    batch = data.get("results", [])
    if not batch:
        break

    raw_events.extend(batch)
    total = data.get("total_count", 0)
    print(f"Fetched {len(raw_events)}/{min(total, MAX_RECORDS)} events")

    if len(raw_events) >= min(total, MAX_RECORDS):
        break

    offset += PAGE_SIZE
    time.sleep(POLITENESS_DELAY)

print(f"\nÉvénements récupérés : {len(raw_events)}")

Fetched 100/10000 events
Fetched 200/10000 events
Fetched 300/10000 events
Fetched 400/10000 events
Fetched 500/10000 events
Fetched 600/10000 events
Fetched 700/10000 events
Fetched 800/10000 events
Fetched 900/10000 events
Fetched 1000/10000 events
Fetched 1100/10000 events
Fetched 1200/10000 events
Fetched 1300/10000 events
Fetched 1400/10000 events
Fetched 1500/10000 events
Fetched 1600/10000 events
Fetched 1700/10000 events
Fetched 1800/10000 events
Fetched 1900/10000 events
Fetched 2000/10000 events
Fetched 2100/10000 events
Fetched 2200/10000 events
Fetched 2300/10000 events
Fetched 2400/10000 events
Fetched 2500/10000 events
Fetched 2600/10000 events
Fetched 2700/10000 events
Fetched 2800/10000 events
Fetched 2900/10000 events
Fetched 3000/10000 events
Fetched 3100/10000 events
Fetched 3200/10000 events
Fetched 3300/10000 events
Fetched 3400/10000 events
Fetched 3500/10000 events
Fetched 3600/10000 events
Fetched 3700/10000 events
Fetched 3800/10000 events
Fetched 3900/10000 ev

## 2. Sauvegarde des données brutes

On sauvegarde avec la date de téléchargement dans le nom de fichier  
pour pouvoir comparer les snapshots au fil du temps.

In [3]:
region_slug = OPENAGENDA_REGION.lower().replace("-", "").replace(" ", "_")
raw_path = DATA_RAW / f"openagenda_{region_slug}_{REFERENCE_DATE.replace('-', '')}.json"

with open(raw_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "reference_date": REFERENCE_DATE,
            "region": OPENAGENDA_REGION,
            "date_from": DATE_FROM,
            "date_to": DATE_TO,
            "total_events": len(raw_events),
            "events": raw_events,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Sauvegardé → {raw_path}")
print(f"Taille     : {raw_path.stat().st_size / 1024 / 1024:.1f} Mo")

Sauvegardé → /Users/kilian/Documents/Repositories/OpenClassrooms/OC-Concevez-et-deployez-un-systeme-RAG/data/raw/openagenda_îledefrance_20260607.json
Taille     : 44.5 Mo


## 3. Chargement dans un DataFrame

In [4]:
df = pd.DataFrame(raw_events)
print(f"Shape    : {df.shape}")
print(f"Colonnes : {df.columns.tolist()}")

Shape    : (10000, 56)
Colonnes : ['uid', 'slug', 'canonicalurl', 'title_fr', 'description_fr', 'longdescription_fr', 'conditions_fr', 'keywords_fr', 'image', 'imagecredits', 'thumbnail', 'originalimage', 'updatedat', 'daterange_fr', 'firstdate_begin', 'firstdate_end', 'lastdate_begin', 'lastdate_end', 'timings', 'accessibility', 'accessibility_label_fr', 'location_uid', 'location_coordinates', 'location_name', 'location_address', 'location_district', 'location_insee', 'location_postalcode', 'location_city', 'location_department', 'location_region', 'location_countrycode', 'location_image', 'location_imagecredits', 'location_phone', 'location_website', 'location_links', 'location_tags', 'location_description_fr', 'location_access_fr', 'attendancemode', 'onlineaccesslink', 'status', 'age_min', 'age_max', 'originagenda_title', 'originagenda_uid', 'contributor_email', 'contributor_contactnumber', 'contributor_contactname', 'contributor_contactposition', 'contributor_organization', 'catego

In [5]:
df.head(5)

,uid,slug,canonicalurl,title_fr,description_fr,longdescription_fr,conditions_fr,keywords_fr,image,imagecredits,...,originagenda_uid,contributor_email,contributor_contactnumber,contributor_contactname,contributor_contactposition,contributor_organization,category,country_fr,registration,links
0,6670838,visite-atelier-prendre-la-mesure-du-temps-140829,https://openagenda.com/jep-2025-ile-de-france/...,Visite-atelier : Prendre la mesure du temps,Atelier créatif,<p>Cette visite-atelier invite petits et grand...,25 participants maximum.,None,https://cdn.openagenda.com/main/ff63a4fa6c4140...,"© Musée de La Poste, 2025",...,6375595,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://billetteri...",None
1,96129611,la-fresque-de-la-biodiversite-comprendre-ce-qu...,https://openagenda.com/biblis-en-folie-2025/ev...,La fresque de la Biodiversité : comprendre ce ...,Au travers d'un atelier ludique et collaborati...,<p>Au travers d'un atelier ludique et collabor...,Entrée gratuite à partir de 12 ans / Réservati...,None,None,None,...,3428022,None,None,None,None,None,None,France (Métropole),"[{""type"": ""link"", ""value"": ""https://mediathequ...",None
2,45384394,on-met-les-voiles-871140,https://openagenda.com/rdvj-2025-ile-de-france...,On met les voiles !,"A l'occasion de ""Rendez-vous aux jardins"", le ...","<p>A l'occasion de ""Rendez-vous aux jardins"", ...",None,None,https://cdn.openagenda.com/main/23cdd3e6fb4a45...,©Centre des monuments nationaux,...,4144624,None,None,None,None,None,None,France (Métropole),None,None
3,67909253,jardin-partage-de-la-harengerie-13-rue-de-la-h...,https://openagenda.com/rdvj-2025-ile-de-france...,Visite libre du Jardin partagé de la Harengerie,Exposition d'art brut,"<p>Sur un terrain en friche, les habitants du ...",None,None,https://cdn.openagenda.com/main/040df3eead6e48...,Ville de Pontoise,...,4144624,None,None,None,None,None,None,France (Métropole),None,None
4,17809408,atelier-rempotage-5733116,https://openagenda.com/rdvj-2025-ile-de-france...,Atelier rempotage,Profitez d'un atelier rempotage pour venir ren...,<p>Profitez d'un atelier rempotage pour venir ...,None,None,https://cdn.openagenda.com/main/fff5c3ad923642...,©Jean-Pierre Delagarde / CMN,...,4144624,None,None,None,None,None,None,France (Métropole),None,None


## 4. Exploration — Structure & Qualité des données

### 4.1 Taux de remplissage par colonne

In [6]:
summary = pd.DataFrame(
    {
        "dtype": df.dtypes,
        "non_null": df.notnull().sum(),
        "null_pct": (df.isnull().sum() / len(df) * 100).round(1),
        "unique": df.astype(str).nunique(),
    }
).sort_values(["unique"], ascending=False)

summary

,dtype,non_null,null_pct,unique
uid,object,10000,0.0,10000
canonicalurl,object,10000,0.0,10000
slug,object,10000,0.0,10000
image,object,9445,5.6,9446
thumbnail,object,9445,5.6,9446
originalimage,object,9445,5.6,9446
title_fr,object,9840,1.6,8660
longdescription_fr,object,9351,6.5,8642
updatedat,object,10000,0.0,8640
timings,object,10000,0.0,7605


In [7]:
# Nombre de doublons
n_duplicates = df.duplicated(subset=["uid"]).sum()
print(f"UIDs dupliqués : {n_duplicates} / {len(df)}")

# Regarder ce que sont ces doublons
dupes = df[df.duplicated(subset=["uid"], keep=False)].sort_values("uid")
print(f"\nNombre d'UIDs concernés : {dupes['uid'].nunique()}")
dupes[["uid", "title_fr", "firstdate_begin", "lastdate_begin", "location_city"]].head(20)

UIDs dupliqués : 0 / 10000

Nombre d'UIDs concernés : 0


,uid,title_fr,firstdate_begin,lastdate_begin,location_city


In [8]:
if n_duplicates > 0:
    print("\nExemple d'UID dupliqué :")
    uid_example = dupes["uid"].iloc[0]
    df[df["uid"] == uid_example][
        ["uid", "title_fr", "firstdate_begin", "lastdate_begin", "description_fr"]
    ]

**Problème** : La pagination par offset est sensible aux insertions concurrentes dans la base — un événement ajouté pendant le téléchargement décale les enregistrements et peut être récupéré deux fois sur deux pages consécutives.

**Solution** : Passer à une pagination par tranches temporelles fixes rend les requêtes idempotentes — une insertion n'affecte que la tranche du mois concerné et ne perturbe pas les autres pages déjà récupérées.

### 4.2 Champs textuels retenus pour le RAG

D'après la structure de l'API, les champs candidats sont :

| Champ | Rôle dans le RAG |
|---|---|
| `title_fr` | Titre de l'événement |
| `description_fr` | Description courte |
| `longdescription_fr` | Description longue (HTML) — contenu principal |
| `keywords_fr` | Mots-clés |
| `daterange_fr` | Date en français lisible |
| `location_name`, `location_city`, `location_address` | Lieu |
| `conditions_fr` | Conditions d'accès (prix, inscription...) |

In [9]:
RAG_FIELDS = [
    "title_fr",
    "description_fr",
    "longdescription_fr",
    "keywords_fr",
    # "daterange_fr", # cf 4.5
    "conditions_fr",
    "location_name",
    "location_city",
    "location_address",
    "location_department",
]

rag_summary = summary.loc[summary.index.isin(RAG_FIELDS)]
rag_summary[["non_null", "null_pct", "unique"]]

,non_null,null_pct,unique
title_fr,9840,1.6,8660
longdescription_fr,9351,6.5,8642
description_fr,9840,1.6,7112
location_address,10000,0.0,4861
location_name,10000,0.0,4808
keywords_fr,4033,59.7,3059
conditions_fr,5346,46.5,2457
location_city,9989,0.1,685
location_department,9792,2.1,22


### 4.3 Distribution temporelle

In [10]:
df["firstdate_begin"] = pd.to_datetime(df["firstdate_begin"], errors="coerce", utc=True)
df["month"] = df["firstdate_begin"].dt.to_period("M")

monthly = df.groupby("month").size().reset_index(name="count")
monthly["month"] = monthly["month"].astype(str)

print("Distribution mensuelle :")
print(monthly.sort_values("count", ascending=False).to_string(index=False))

Distribution mensuelle :
  month  count
2025-09   1936
2026-06   1338
2025-10    789
2026-05    772
2025-06    722
2026-04    683
2025-11    654
2025-07    508
2026-02    497
2026-03    472
2026-01    372
2026-09    318
2025-12    310
2026-07    235
2025-08    159
2026-10     70
2026-08     44
2026-11     40
2026-12     20
2027-01     19
2027-03     15
2027-04     11
2027-02      8
2027-05      7
2027-06      1


/var/folders/6_/pkc98_l54jd53vtftrl1lb6w0000gn/T/ipykernel_8797/1710302120.py:2: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"] = df["firstdate_begin"].dt.to_period("M")


In [11]:
now_utc = pd.Timestamp.now(tz="UTC")
n_past = (df["firstdate_begin"] < now_utc).sum()
n_future = (df["firstdate_begin"] >= now_utc).sum()
n_null = df["firstdate_begin"].isnull().sum()

print(f"Événements passés  : {n_past:>6} ({n_past / len(df) * 100:.1f}%)")
print(f"Événements futurs  : {n_future:>6} ({n_future / len(df) * 100:.1f}%)")
print(f"Dates manquantes   : {n_null:>6} ({n_null / len(df) * 100:.1f}%)")

Événements passés  :   9076 (90.8%)
Événements futurs  :    924 (9.2%)
Dates manquantes   :      0 (0.0%)


### 4.4 Distribution géographique

In [12]:
print("Top 15 villes :")
print(df["location_city"].value_counts().head(15).to_string())

print("\nRépartition par département :")
print(df["location_department"].value_counts().to_string())

Top 15 villes :
location_city
Paris                      3218
Issy-les-Moulineaux         230
Fontainebleau               211
Versailles                  193
Jouy-en-Josas               170
Saint-Ouen-sur-Seine        130
Montreuil                   126
Saint-Denis                 119
Saint-Germain-en-Laye       112
Le Chesnay-Rocquencourt     100
Gif-sur-Yvette               91
Boulogne-Billancourt         90
Cergy                        83
Louvres                      83
Courbevoie                   79

Répartition par département :
location_department
Paris                3042
Yvelines             1459
Hauts-de-Seine       1055
Seine-et-Marne       1051
Seine-Saint-Denis     982
Val-d'Oise            836
Essonne               807
Val-de-Marne          494
Seine-St-Denis         19
Seine-St.-Denis        17
Val-D'Oise             10
Île-de-France           4
Val-De-Marne            3
75012                   3
Métropole de Lyon       2
paris                   2
75008                  

### 4.5 `daterange_fr`

Il faut regarde ce que contient ce champs pour voir s'il correspond aux dates qu'on utilise pour requêter l'API. Ce champs peut-être imprécis ou même faux et donc pourrait induire en erreur le RAG.

In [13]:
print("=== Exemples de daterange_fr ===")
print(df["daterange_fr"].dropna().sample(20, random_state=42).to_string())

=== Exemples de daterange_fr ===
6252              20 et 21 septembre 2025
4684    Dimanche 21 septembre 2025, 14h00
1731             Mardi 18 mai 2027, 20h30
4742      Samedi 20 septembre 2025, 14h00
4521                 Mardi 16 juin, 09h30
6340              20 et 21 septembre 2025
576                   Mardi 3 mars, 19h30
5202             Samedi 10 janvier, 21h00
6363                 Samedi 6 juin, 12h00
439      Mercredi 19 novembre 2025, 18h30
2750          Jeudi 9 octobre 2025, 18h30
7487                 Samedi 23 mai, 19h15
5272              Mercredi 1 avril, 16h00
5653         Dimanche 20 septembre, 10h30
3999              20 et 21 septembre 2025
6033                 Lundi 29 juin, 09h30
582                  Jeudi 19 mars, 11h45
9930                  4 et 5 octobre 2025
7051              20 et 21 septembre 2025
8158                 Samedi 23 mai, 15h30


In [14]:
compare_cols = ["title_fr", "daterange_fr", "firstdate_begin", "lastdate_begin"]

df[compare_cols].dropna(subset=["daterange_fr"]).sample(10, random_state=42)

,title_fr,daterange_fr,firstdate_begin,lastdate_begin
6252,Visite du domaine de Meudon,20 et 21 septembre 2025,2025-09-20 13:00:00+00:00,2025-09-21T13:00:00+00:00
4684,Ferme nomade des bergeries d'Issy,"Dimanche 21 septembre 2025, 14h00",2025-09-21 12:00:00+00:00,2025-09-21T12:00:00+00:00
1731,HOMMAGE A MICHEL BERGER,"Mardi 18 mai 2027, 20h30",2027-05-18 18:30:00+00:00,2027-05-18T18:30:00+00:00
4742,"Conférence ""Les astrolabes, leur histoire et l...","Samedi 20 septembre 2025, 14h00",2025-09-20 12:00:00+00:00,2025-09-20T12:00:00+00:00
4521,Réunion d'information Pôle d'orientation Netto...,"Mardi 16 juin, 09h30",2026-06-16 07:30:00+00:00,2026-06-16T07:30:00+00:00
6340,"Histoire, gastronomie et culture au Ministère ...",20 et 21 septembre 2025,2025-09-20 10:00:00+00:00,2025-09-21T08:00:00+00:00
576,Le ciné-quiz de L'Epée 🔍,"Mardi 3 mars, 19h30",2026-03-03 18:30:00+00:00,2026-03-03T18:30:00+00:00
5202,Je sors ce soir de Guillaume Dustan,"Samedi 10 janvier, 21h00",2026-01-10 20:00:00+00:00,2026-01-10T20:00:00+00:00
6363,Village des start up Arènes de Lutèce - Les mi...,"Samedi 6 juin, 12h00",2026-06-06 10:00:00+00:00,2026-06-06T10:00:00+00:00
439,Afterwork ADP / AIR FRANCE - Dégustation de vins,"Mercredi 19 novembre 2025, 18h30",2025-11-19 17:30:00+00:00,2025-11-19T17:30:00+00:00


In [15]:
mask_incoherent = (df["daterange_fr"].notnull() & df["firstdate_begin"].isnull()) | (
    df["daterange_fr"].isnull() & df["firstdate_begin"].notnull()
)

print(f"Événements avec incohérence daterange_fr / firstdate_begin : {mask_incoherent.sum()}")
df[mask_incoherent][compare_cols].head(10)

Événements avec incohérence daterange_fr / firstdate_begin : 0


,title_fr,daterange_fr,firstdate_begin,lastdate_begin


C'est ce qu'on pouvait craindre. Alors déjà toutes les données sont parfaitement cohérente, donc ça, ça va. Mais parfois les données sont imprécises sur cette colonne : il faudrait mieux créer une colonne qui soit plus régulière pour ces informations.

### 4.6 Longueur des descriptions

Point critique pour choisir la **stratégie de chunking** dans le notebook 03.

In [16]:
df["desc_short_len"] = df["description_fr"].fillna("").str.len()
df["desc_long_len"] = df["longdescription_fr"].fillna("").apply(html_to_text).str.len()

print("=== description_fr (courte) ===")
print(df["desc_short_len"].describe().round(0))
print(f"Vides : {(df['desc_short_len'] == 0).sum()}")

print("\n=== longdescription_fr (longue, nettoyée via Docling) ===")
print(df["desc_long_len"].describe().round(0))
print(f"Vides : {(df['desc_long_len'] == 0).sum()}")

Clashing formatting: 'bold=True italic=True underline=False strikethrough=False script=<Script.BASELINE: 'baseline'>' and 'bold=True italic=False underline=False strikethrough=False script=<Script.BASELINE: 'baseline'>'! Chose 'bold=True italic=False underline=False strikethrough=False script=<Script.BASELINE: 'baseline'>'


=== description_fr (courte) ===
count    10000.0
mean        86.0
std         66.0
min          0.0
25%         30.0
50%         66.0
75%        144.0
max        200.0
Name: desc_short_len, dtype: float64
Vides : 160

=== longdescription_fr (longue, nettoyée via Docling) ===
count    10000.0
mean       712.0
std        746.0
min          0.0
25%        263.0
50%        500.0
75%        928.0
max       9857.0
Name: desc_long_len, dtype: float64
Vides : 649


In [17]:
bins = [0, 1, 200, 500, 1000, 2000, 5000, float("inf")]
labels = ["vide", "<200", "200-500", "500-1k", "1k-2k", "2k-5k", ">5k"]

df["desc_bucket"] = pd.cut(df["desc_long_len"], bins=bins, labels=labels)

print("Distribution des descriptions longues par tranche (caractères) :")
print(df["desc_bucket"].value_counts().sort_index().to_string())

Distribution des descriptions longues par tranche (caractères) :
desc_bucket
vide          0
<200       1243
200-500    3162
500-1k     2731
1k-2k      1730
2k-5k       446
>5k          39


### 4.7 Exemple d'événement complet — champs RAG

In [18]:
sample = df[df["desc_long_len"] > 500].iloc[0]

for field in RAG_FIELDS + ["firstdate_begin", "lastdate_begin", "canonicalurl"]:
    val = sample.get(field, "N/A")
    if not (pd.isnull(val) if not isinstance(val, (list, dict)) else False) and val != "":
        display_val = html_to_text(str(val)) if "description" in field else str(val)
        print(f"{field:<30}: {display_val}")

title_fr                      : Visite commentée du Parc de Courances
description_fr                : Proche de Paris, Courances est l'un des plus beaux jardins de France et habite un château Louis XIII, classés au titre des monuments historiques. Le parc de Courances comme le château sont à l'image...
longdescription_fr            : Proche de Paris, Courances est l'un des plus beaux jardins de France et habite un château Louis XIII, classés au titre des monuments historiques. Le parc de Courances comme le château sont à l'image des différentes époques et propriétaires qui les ont façonnés. Ainsi cohabitent harmonieusement un jardin d'eau de la Renaissance, un parc classique « à la française » et un jardin japonais. L'ensemble est alimenté par 14 sources et ponctué par 17 pièces d'eau.

Ce parc d'une surprenante beauté, classique et romantique, historique et contemporaine, recèle une atmosphère à la fois intime et grandiose. L'omniprésence des eaux "courantes", du végétal et de la pier

## 5. Bilan & décisions pour la suite

| Question | Observation |
|---|---|
| Volume récupéré | 10 000 évènements (44.0 Mo) |
| Date de référence | `REFERENCE_DATE` — fenêtre ±1 an automatique |
| Champ texte principal retenu | `longdescription_fr` (HTML → à stripper via Docling) |
| Taux de descriptions vides | description courte vide : 1.7% — description longue vide : 7.3% à éliminer |
| Longueur médiane des descriptions | 65 et 500 caractères |
| Nettoyages nécessaires | strip HTML (Docling), doublons sur `uid`, normaliser les dates |

---
**Suite** : `02.0-kl-preprocessing.ipynb` — nettoyage